In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

from google.colab import drive
drive.mount('/content/drive')

# Load your hotel dataset (replace with actual path)
df = pd.read_csv("/content/drive/MyDrive/MLops/hotels_data_cleaned.csv")

# Assume last column is the target (hotel price)
X = df.iloc[:, :-1]  # All features except the target
y = df.iloc[:, -1]   # The target is the last column (hotel price)

column_name = df.columns[-1]
print(f"The name of the target column is: {column_name}")

# Split into train/test sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Define regression models
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Support Vector Regressor": SVR(),
    "XGBoost Regressor": XGBRegressor(objective='reg:squarederror', random_state=42)
}

# Store results
results = []

# Evaluate each model
for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    end = time.time()

    # Avoid division by zero in MAPE
    non_zero_indices = y_test != 0
    y_test_non_zero = y_test[non_zero_indices]
    y_pred_non_zero = y_pred[non_zero_indices]
    mape = np.mean(np.abs((y_test_non_zero - y_pred_non_zero) / y_test_non_zero)) * 100

    results.append({
        "Model": name,
        "R² Score": round(r2_score(y_test, y_pred), 4),
        "MAE": round(mean_absolute_error(y_test, y_pred), 4),
        "MAPE (%)": round(mape, 2),
        "Train Time (s)": round(end - start, 3)
    })

# Convert results to DataFrame and sort by MAE (mean absolute error)
results_df = pd.DataFrame(results).sort_values(by="MAE")

# Display model evaluation results
print("\n🔍 Model Evaluation Summary:")
print(results_df.to_string(index=False))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
The name of the target column is: Guest reviews

🔍 Model Evaluation Summary:
                   Model  R² Score    MAE  MAPE (%)  Train Time (s)
        Ridge Regression   -0.0157 0.1745      3.75           0.007
           Decision Tree    0.1100 0.1833      3.99           0.003
Support Vector Regressor   -0.2794 0.1921      4.08           0.004
           Random Forest   -0.1624 0.1938      4.16           0.146
        Lasso Regression   -0.2614 0.2188      4.72           0.003
       XGBoost Regressor   -0.5780 0.2230      4.80           0.096
       Linear Regression   -1.4038 0.2582      5.72           0.022


In [ ]:
import pandas as pd
import numpy as np
import time
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder
from google.colab import drive

# 📂 Mount Google Drive
drive.mount('/content/drive')

# 📂 Load the Cleaned Hotel Dataset
csv_path = "/content/drive/MyDrive/MLops/hotels_data_cleaned.csv"  # Replace with actual path
print(f"\n📂 Loading data from: {csv_path}")
df = pd.read_csv(csv_path)

# 🧹 Drop unnecessary columns
df.drop(["Booking Link", "Description", "Most Popular Facilities",
         "Room type & Number of guests", "Your choices",
         "Hotel area info", "Guest reviews"], axis=1, inplace=True)

# 🔠 Encode categorical columns (e.g., Room Type, Location, etc.)
label_cols = ["Room Type", "Location", "Property Type"]  # Adjust based on dataset
le_dict = {}
for col in label_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le  # Store encoders for later use

# Separate features and target variable
X = df.drop("Price", axis=1)  # Assuming 'Price' is the target column
y = df["Price"]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ✅ Feature processing complete.
print("✅ Feature processing complete.")
print("Shape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)

# 📦 Define regression models
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Support Vector Regressor": SVR(),
    "XGBoost Regressor": XGBRegressor(objective='reg:squarederror', random_state=42)
}

# Store results of model evaluation
results = []

# Evaluate each model
for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    end = time.time()

    # Avoid division by zero in MAPE
    non_zero_indices = y_test != 0
    y_test_non_zero = y_test[non_zero_indices]
    y_pred_non_zero = y_pred[non_zero_indices]
    mape = np.mean(np.abs((y_test_non_zero - y_pred_non_zero) / y_test_non_zero)) * 100

    results.append({
        "Model": name,
        "R² Score": round(r2_score(y_test, y_pred), 4),
        "MAE": round(mean_absolute_error(y_test, y_pred), 2),
        "MAPE (%)": round(mape, 2),
        "Train Time (s)": round(end - start, 3)
    })

# Convert results to DataFrame and sort by MAE
results_df = pd.DataFrame(results).sort_values(by="MAE")

# Display model evaluation summary
print("\n🔍 Model Evaluation Summary for Hotel Price Prediction:")
print(results_df.to_string(index=False))

# 🔝 Train the best model (Random Forest) again
best_model = RandomForestRegressor(random_state=42)
best_model.fit(X_train, y_train)

# Save the trained model
joblib.dump(best_model, "/content/drive/MyDrive/MLops/hotel_price_predictor.pkl")
print()
print("✅ Model saved as 'hotel_price_predictor.pkl'")

# 📂 Load the trained model
model = joblib.load("/content/drive/MyDrive/MLops/hotel_price_predictor.pkl")

# 📂 Load new hotel data (without 'Price' column)
new_data_path = "/content/drive/MyDrive/MLops/hotels_data_cleaned.csv"  # Replace with actual path
new_data = pd.read_csv(new_data_path)

# One-hot encode the new data and align columns with the training data
X_new = pd.get_dummies(new_data)
X_new = X_new.reindex(columns=X_train.columns, fill_value=0)

# Predict the prices using the trained model
predicted_prices = model.predict(X_new)

# Add predictions to the new data
new_data["Predicted Price"] = predicted_prices

# Show top rows of the result
print(new_data.head())

# 📂 Save predictions to a new CSV
new_data.to_csv("/content/drive/MyDrive/MLops/hotel_predicted_prices.csv", index=False)
print()
print("✅ Predicted prices saved to 'hotel_predicted_prices.csv'")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📂 Loading data from: /content/drive/MyDrive/MLops/hotels_data_cleaned.csv
✅ Feature processing complete.
Shape of X_train: (46, 12)
Shape of y_train: (46,)

🔍 Model Evaluation Summary for Hotel Price Prediction:
                   Model  R² Score  MAE  MAPE (%)  Train Time (s)
           Decision Tree    0.0891 2.83     98.66           0.005
        Ridge Regression    0.0328 2.88    192.38           0.003
           Random Forest    0.1154 3.01    153.43           0.212
Support Vector Regressor   -0.0227 3.08    203.01           0.007
        Lasso Regression    0.0872 3.09    180.98           0.004
       Linear Regression   -0.1024 3.12    182.73           0.005
       XGBoost Regressor    0.0011 3.21    164.67           0.202

✅ Model saved as 'hotel_price_predictor.pkl'
   Rating  Price  Location  Distance from City Center  Booking Link  \
0       5    